# Web Search Tools and Multi-Hop Retrieval (Chapter 8)

This notebook accompanies Chapter 8 §8.3.2 and §8.3.3 of *Context Engineering with DSPy*. You will wrap the Serper web-search API as a DSPy tool, give it to a ReAct agent alongside a knowledge-base search, and then compare two approaches to multi-hop retrieval: letting ReAct decide when to search again versus a fixed-depth custom `MultiHop` module.

**Required environment variables**

- `OPENAI_API_KEY` — used by the default `openai/gpt-5-mini` LM.
- `SERPER_API_KEY` — get a free key at <https://serper.dev>. The notebook checks for it before any web call is made.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))

## Environment check

Confirm both API keys are present. The web-search tool short-circuits with a clear message rather than crashing if `SERPER_API_KEY` is missing.

In [ ]:
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

if not SERPER_API_KEY:
    print(
        "SERPER_API_KEY is not set. Get a key at https://serper.dev and add it to your .env "
        "before running the web_search cell."
    )
else:
    print("SERPER_API_KEY found.")

## Wrapping a web search API as a DSPy tool

Serper returns Google-style organic results. We collapse the top three snippets into a single newline-separated string — that is all the agent needs to read.

In [ ]:
import requests


def web_search(query: str) -> str:
    """Search the web for current information about a topic."""
    if not SERPER_API_KEY:
        return "web_search unavailable: set SERPER_API_KEY in your environment."
    response = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
        json={"q": query},
        timeout=20,
    )
    response.raise_for_status()
    results = response.json().get("organic", [])
    return "\n".join(r.get("snippet", "") for r in results[:3])


def search_knowledge_base(query: str) -> str:
    """Search our internal product documentation."""
    return f"[stub] internal docs hit for: {query}"


def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))

## ReAct with mixed retrieval tools

Give the agent both an internal knowledge-base search and the public web. It learns to route between them from the tool descriptions alone.

In [ ]:
agent = dspy.ReAct(
    "question -> answer",
    tools=[search_knowledge_base, web_search, calculate],
    max_iters=10,
)

result = agent(question="What is the most recent DSPy release on PyPI?")
print(result.answer)

## Approach 1: multi-hop with ReAct

The simplest multi-hop strategy: hand the agent a single search tool with a generous `max_iters` budget and let it decide when to search again.

In [ ]:
def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return short snippets for the top results."""
    if not SERPER_API_KEY:
        return "search_wikipedia unavailable: set SERPER_API_KEY in your environment."
    response = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
        json={"q": f"site:en.wikipedia.org {query}"},
        timeout=20,
    )
    response.raise_for_status()
    results = response.json().get("organic", [])
    return "\n".join(f"{r.get('title', '')}: {r.get('snippet', '')}" for r in results[:5])


multihop_agent = dspy.ReAct(
    "claim -> titles: list[str]",
    tools=[search_wikipedia],
    max_iters=12,
)

## Approach 2: custom fixed-depth `MultiHop` module

When you want a predictable number of hops — useful for optimization and for forcing the agent to gather more evidence — wrap the loop in a `dspy.Module`. Each hop generates a query, retrieves passages, and integrates the new context into running notes.

In [ ]:
def search(query: str, k: int = 10) -> list[str]:
    """Return up to k passages for the query."""
    if not SERPER_API_KEY:
        return ["search unavailable: set SERPER_API_KEY in your environment."]
    response = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
        json={"q": query, "num": k},
        timeout=20,
    )
    response.raise_for_status()
    results = response.json().get("organic", [])
    return [f"{r.get('title', '')}: {r.get('snippet', '')}" for r in results[:k]]


class MultiHop(dspy.Module):
    def __init__(self, num_hops=3):
        super().__init__()
        self.num_hops = num_hops
        self.generate_query = dspy.ChainOfThought("claim, notes -> query")
        self.integrate = dspy.ChainOfThought(
            "claim, notes, context -> new_notes: list[str], titles: list[str]"
        )

    def forward(self, claim):
        notes, titles = [], []
        for _ in range(self.num_hops):
            query = self.generate_query(claim=claim, notes=notes).query
            context = search(query, k=10)
            pred = self.integrate(claim=claim, notes=notes, context=context)
            notes.extend(pred.new_notes)
            titles.extend(pred.titles)
        return dspy.Prediction(titles=list(set(titles)))